# 수집 결과 병합

곡 목록(순위 노트북의 최종 출력)에 가사·장르 수집 노트북과 특징 추출 노트북 3개의 결과를 **`video_id` 기준으로 붙여** 하나의 표로 만드는 노트북입니다. 이 표가 이후 전처리(곡 단위 중복 제거 등)와 분석의 출발점입니다.

## 구성

| 섹션 | 역할 | 입력 | 출력 |
|---|---|---|---|
| **0. 설정** | 병합할 파일과 각 파일에서 가져올 컬럼 정의 | — | — |
| **1. 병합** | 곡 목록에 결과 파일 4개의 새 컬럼을 `video_id`로 붙이고 검사·저장 | 곡 목록 + 결과 파일 4개 | `kpop_radar_2023_2025_dedup_merged_features.csv` |

실행 순서는 0 → 1입니다.

## 병합 대상

| 이름 | 파일 | 가져오는 컬럼 |
|---|---|---|
| (곡 목록) | `kpop_radar_2023_2025_dedup_with_youtubeapi.csv` | 병합의 기준. `songName`, `artists`, `url`, `video_id`, 조회수 등 |
| `genre_lyrics` | `kpop_radar_2023_2025_dedup_with_genre_lyrics.csv` | `genie_genre`, `genie_lyrics` |
| `visual` | `kpop_radar_2023_2025_dedup_with_visual.csv` | `avg_brightness`, `avg_motion`, `avg_r_value`, `avg_g_value`, `avg_b_value`, `frames_analyzed`, `video_duration`, `video_fps` + `status` |
| `audio_valence_arousal` | `kpop_radar_2023_2025_dedup_with_audio_valence_arousal.csv` | `valence_raw`, `arousal_raw`, `valence_normalized`, `arousal_normalized` + `status` |
| `audio_additional` | `kpop_radar_2023_2025_dedup_with_audio_additional.csv` | `energy`, `energy_variance`, `loudness`, `tempo`, `duration_ms`, `speechiness`, `speech_duration_sec`, `spectral_flatness`, `harmonic_ratio`, `acoustic_score` + `status` |

**병합 방식과 참고사항**
- **왼쪽 병합**입니다. 곡 목록의 모든 곡이 그대로 남고, 특징 추출에 실패했거나 결과 파일에 없는 곡은 해당 컬럼이 빈 값이 됩니다.
- 결과 파일은 입력 컬럼을 모두 갖고 있지만, 병합에는 **`video_id`와 그 노트북이 새로 만든 컬럼만** 사용합니다(컬럼이 겹치지 않게).
- 각 노트북의 `status` 컬럼은 `<이름>_status`(예: `visual_status`)로 이름을 바꿔 남겨 두어, 어떤 곡이 어디서 실패했는지 추적할 수 있습니다. `genre_lyrics`는 `status`가 없어서 값이 비었는지로 판단합니다.
- 코드가 자동으로 검사하는 것: 곡 목록과 각 결과 파일의 `video_id`가 유일한지, 새 컬럼이 곡 목록의 컬럼과 겹치지 않는지, 병합 후 행 수가 곡 목록과 같은지. 결과 파일이 하나라도 없으면 어떤 파일이 없는지 알려 주고 멈춥니다.
- 앞 노트북(`crawling_kpop_rankings_youtube_metadata`, `crawling_genre_lyrics`, `feature_extraction_*` 3개)이 모두 실행되어 결과 파일이 `../data/raw/`에 있어야 합니다. 결과 파일과 병합 결과에는 가사 원문 등 저작권이 있는 정보가 들어 있어 이 저장소에 포함하지 않았습니다.


## 0. 설정

병합할 파일과 각 파일에서 가져올 컬럼을 정의합니다. **가장 먼저 실행하세요.**

- `DATA_DIR` — 결과 파일이 있는 폴더(`data_preprocessing/` 기준 `../data/raw/`, 저장소에는 포함하지 않음)
- `BASE_FILE` — 병합의 기준이 되는 곡 목록
- `SOURCES` — 병합할 결과 파일 목록. 각 항목은 이름(`name`), 파일(`file`), 가져올 컬럼(`columns`), `status` 컬럼 이름(`status`, 없으면 `None`)입니다.
- `OUTPUT_FILE` — 병합 결과 파일

In [ ]:
# 공통 설정: 병합할 파일과 각 파일에서 가져올 컬럼
from pathlib import Path

DATA_DIR = Path("../data/raw")                    # 수집 결과가 있는 폴더 (.gitignore 대상)

# 병합의 기준이 되는 곡 목록 (순위 노트북의 최종 출력, video_id가 유일)
BASE_FILE = DATA_DIR / "kpop_radar_2023_2025_dedup_with_youtubeapi.csv"

# 병합할 결과 파일: 각 노트북이 새로 만든 컬럼만 가져온다
SOURCES = [
    dict(name="genre_lyrics",
         file=DATA_DIR / "kpop_radar_2023_2025_dedup_with_genre_lyrics.csv",
         columns=['genie_genre', 'genie_lyrics'],
         status=None),
    dict(name="visual",
         file=DATA_DIR / "kpop_radar_2023_2025_dedup_with_visual.csv",
         columns=['avg_brightness', 'avg_motion', 'avg_r_value', 'avg_g_value', 'avg_b_value',
                  'frames_analyzed', 'video_duration', 'video_fps'],
         status='status'),
    dict(name="audio_valence_arousal",
         file=DATA_DIR / "kpop_radar_2023_2025_dedup_with_audio_valence_arousal.csv",
         columns=['valence_raw', 'arousal_raw', 'valence_normalized', 'arousal_normalized'],
         status='status'),
    dict(name="audio_additional",
         file=DATA_DIR / "kpop_radar_2023_2025_dedup_with_audio_additional.csv",
         columns=['energy', 'energy_variance', 'loudness', 'tempo', 'duration_ms',
                  'speechiness', 'speech_duration_sec',
                  'spectral_flatness', 'harmonic_ratio', 'acoustic_score'],
         status='status'),
]

OUTPUT_FILE = DATA_DIR / "kpop_radar_2023_2025_dedup_merged_features.csv"   # 병합 결과

## 1. 병합

1. 곡 목록(`BASE_FILE`)과 결과 파일 4개를 읽고, 결과 파일이 모두 있는지 확인합니다.
2. 각 결과 파일에서 `video_id`와 새 컬럼(과 `status`)만 꺼내 곡 목록에 `video_id` 기준으로 왼쪽 병합합니다.
3. 파일마다 값이 채워진 곡 수와 비어 있는 곡 수를 출력합니다.
4. 모든 특징이 채워진 곡 수를 확인하고 `OUTPUT_FILE`로 저장합니다.

병합할 때마다 `video_id`의 유일성, 컬럼 겹침, 행 수 유지를 검사하므로 문제가 있으면 바로 오류로 멈춥니다.

In [ ]:
# 1. 병합: 곡 목록에 결과 파일의 새 컬럼을 video_id 기준으로 붙이고(왼쪽 병합) 저장
import pandas as pd


def load_source(source):
    """결과 파일에서 video_id와 그 노트북이 새로 만든 컬럼(과 status)만 꺼낸다."""
    df = pd.read_csv(source["file"], encoding='utf-8-sig')
    assert df['video_id'].is_unique, f"{source['name']}: video_id가 유일하지 않습니다"

    keep = ['video_id'] + source["columns"]
    if source["status"]:
        keep.append(source["status"])
    part = df[keep].copy()

    # status 컬럼은 어느 노트북의 것인지 알 수 있게 이름을 바꿔 남긴다
    if source["status"]:
        part = part.rename(columns={source["status"]: f"{source['name']}_status"})
    return part


# ============================================
# 곡 목록과 결과 파일 확인
# ============================================
missing_files = [str(s["file"]) for s in SOURCES if not s["file"].exists()]
if missing_files:
    raise FileNotFoundError("결과 파일이 없습니다. 앞 노트북을 먼저 실행하세요:\n  " + "\n  ".join(missing_files))

base = pd.read_csv(BASE_FILE, encoding='utf-8-sig')
assert base['video_id'].is_unique, "곡 목록의 video_id가 유일하지 않습니다"
print(f"📂 곡 목록: {len(base):,}곡 ({BASE_FILE.name})\n")

# ============================================
# 결과 파일 병합 (왼쪽 병합)
# ============================================
merged = base
for source in SOURCES:
    part = load_source(source)

    overlap = (set(part.columns) - {'video_id'}) & set(merged.columns)
    assert not overlap, f"{source['name']}: 이미 있는 컬럼과 겹칩니다 {sorted(overlap)}"

    outside = (~part['video_id'].isin(base['video_id'])).sum()
    merged = merged.merge(part, on='video_id', how='left', validate='one_to_one')
    assert len(merged) == len(base), f"{source['name']}: 병합 후 행 수가 달라졌습니다"

    filled = merged[source["columns"]].notna().all(axis=1).sum()
    print(f"🔗 {source['name']:22s} 결과 {len(part):>5,}곡 | 값 채워짐 {filled:>5,}곡 | 비어 있음 {len(base) - filled:>4,}곡"
          + (f" | 곡 목록에 없는 video_id {outside}개" if outside else ""))

# ============================================
# 결과 확인 및 저장
# ============================================
feature_cols = [c for s in SOURCES for c in s["columns"]]
complete = merged[feature_cols].notna().all(axis=1)
print(f"\n✅ 병합 완료: {len(merged):,}곡 × {merged.shape[1]}컬럼")
print(f"   모든 특징이 채워진 곡: {complete.sum():,}곡 / 하나라도 빈 곡: {(~complete).sum():,}곡")

merged.to_csv(OUTPUT_FILE, index=False, encoding='utf-8-sig')
print(f"📁 저장 위치: {OUTPUT_FILE}")